In [1]:
import os
import re
import wave
import struct
import subprocess
import tempfile
from pathlib import Path

import numpy as np

from tqdm.notebook import tqdm


DATA_DIR = Path('data')


def _sorted_timestamps(directory: Path, suffix: str) -> list[tuple[float, Path]]:
    files = sorted(directory.glob(f'*{suffix}'), key=lambda p: float(p.stem))
    return [(float(p.stem), p) for p in files]


def _write_image_concat(frames: list[tuple[float, Path]], tmp_dir: str) -> str:
    concat_path = os.path.join(tmp_dir, 'images.txt')
    with open(concat_path, 'w') as f:
        for i, (ts, path) in enumerate(frames):
            duration = frames[i + 1][0] - ts if i + 1 < len(frames) else 1.0 / 25.0
            f.write(f"file '{path.resolve()}'\n")
            f.write(f'duration {duration:.6f}\n')
        f.write(f"file '{frames[-1][1].resolve()}'\n")
    return concat_path


def _merge_audio(chunks: list[tuple[float, Path]], t0: float, tmp_dir: str) -> str:
    sample_rate = 24000
    all_samples = []
    params = None
    for _, path in tqdm(chunks, desc='  merging audio', leave=False):
        with wave.open(str(path), 'rb') as wf:
            if params is None:
                params = wf.getparams()
            raw = wf.readframes(wf.getnframes())
        all_samples.append(np.frombuffer(raw, dtype=np.int16))
    audio = np.concatenate(all_samples).astype(np.float32)
    peak = np.abs(audio).max()
    if peak > 0:
        audio = audio / peak * 32000
    audio = audio.astype(np.int16)
    # Prepend silence so audio t0 aligns with image t0
    t0_audio = float(chunks[0][1].stem)
    leading_silence_frames = max(0, int((t0_audio - t0) * sample_rate))
    if leading_silence_frames > 0:
        silence = np.zeros(leading_silence_frames, dtype=np.int16)
        audio = np.concatenate([silence, audio])
    out_path = os.path.join(tmp_dir, 'merged.wav')
    with wave.open(out_path, 'wb') as out:
        out.setparams(params)
        out.writeframes(audio.tobytes())
    return out_path


def _run_ffmpeg_with_progress(cmd: list[str], total_frames: int, desc: str) -> tuple[int, str]:
    """
    Run ffmpeg with -progress pipe:1 and update a tqdm bar based on
    the 'frame=' lines it emits. Returns (returncode, stderr).
    """
    progress_cmd = cmd + ['-progress', 'pipe:1', '-nostats']
    bar = tqdm(total=total_frames, desc=desc, unit='frame', leave=False)
    stderr_lines = []
    last_frame = 0

    with subprocess.Popen(
        progress_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    ) as proc:
        for line in proc.stdout:
            m = re.match(r'frame=(\d+)', line)
            if m:
                current = int(m.group(1))
                bar.update(current - last_frame)
                last_frame = current
        stderr_lines = proc.stderr.read()
        proc.wait()
        returncode = proc.returncode

    bar.update(total_frames - last_frame)
    bar.close()
    return returncode, stderr_lines


def process_run(run_dir: Path) -> None:
    images_dir = run_dir / 'images'
    audio_dir = run_dir / 'audio'
    output_dir = run_dir / 'processed'
    output_dir.mkdir(exist_ok=True)
    output_path = output_dir / '0.mp4'

    frames = _sorted_timestamps(images_dir, '.png')
    if not frames:
        print(f'[{run_dir.name}] No images found, skipping.')
        return

    chunks = _sorted_timestamps(audio_dir, '.wav')

    # Use the later of the two stream start times as t0 so there is no leading silence
    t0_images = frames[0][0]
    t0_audio = chunks[0][0] if chunks else t0_images
    t0 = max(t0_images, t0_audio)

    # Trim frames that precede t0
    frames = [(ts, p) for ts, p in frames if ts >= t0]
    if not frames:
        print(f'[{run_dir.name}] No frames after t0, skipping.')
        return

    with tempfile.TemporaryDirectory() as tmp_dir:
        concat_path = _write_image_concat(frames, tmp_dir)

        cmd = [
            'ffmpeg', '-y',
            '-f', 'concat', '-safe', '0', '-i', concat_path,
        ]

        if chunks:
            merged_wav = _merge_audio(chunks, t0, tmp_dir)
            cmd += ['-i', merged_wav]
            cmd += ['-map', '0:v', '-map', '1:a']
        else:
            cmd += ['-map', '0:v']

        tmp_output = os.path.join(tmp_dir, 'output.mp4')
        cmd += [
            '-c:v', 'libx264', '-pix_fmt', 'yuv420p',
            '-c:a', 'aac', '-b:a', '128k',
            '-af', 'aresample=async=1',
            '-vsync', 'vfr',
            tmp_output,
        ]

        returncode, stderr = _run_ffmpeg_with_progress(
            cmd,
            total_frames=len(frames),
            desc=f'  encoding [{run_dir.name}]',
        )

        if returncode != 0:
            print(f'[{run_dir.name}] ffmpeg failed:\n{stderr}')
        else:
            import shutil
            shutil.copy2(tmp_output, str(output_path))
            print(f'[{run_dir.name}] Written to {output_path}')


run_dirs = sorted(
    [d for d in DATA_DIR.iterdir() if d.is_dir() and d.name.isdigit()],
    key=lambda d: int(d.name),
)

for run_dir in tqdm(run_dirs, desc='runs'):
    process_run(run_dir)


runs:   0%|          | 0/1 [00:00<?, ?it/s]

  merging audio:   0%|          | 0/3568 [00:00<?, ?it/s]

  encoding [0]:   0%|          | 0/1789 [00:00<?, ?frame/s]

[0] Written to data/0/processed/0.mp4


In [2]:
from pathlib import Path
import wave

run_dirs = sorted(
    [d for d in DATA_DIR.iterdir() if d.is_dir() and d.name.isdigit()],
    key=lambda d: int(d.name),
)

for run_dir in run_dirs:
    chunks = sorted((run_dir / 'audio').glob('*.wav'), key=lambda p: float(p.stem))
    if not chunks:
        print(f'[{run_dir.name}] No audio chunks, skipping.')
        continue

    out_path = run_dir / 'processed' / '0.wav'
    out_path.parent.mkdir(exist_ok=True)

    with wave.open(str(out_path), 'wb') as out:
        for i, path in enumerate(chunks):
            with wave.open(str(path), 'rb') as wf:
                if i == 0:
                    out.setparams(wf.getparams())
                out.writeframes(wf.readframes(wf.getnframes()))

    print(f'[{run_dir.name}] Written to {out_path}')

[0] Written to data/0/processed/0.wav
